# Azure AI-103 Labs — Unique Code Snippets Reference

A deduplicated reference of every **distinct** reusable code pattern found across the 21 lab scripts in `14-azure-ai103/Labs/`. Each pattern appears **once**, even though the same idiom (e.g. building a bearer-token credential, connecting to a Foundry agent) is repeated near-identically across several lab files.

Cells are NOT meant to run top-to-bottom as one program — each is an independent snippet lifted from a specific lab (noted above the cell). Fill in your own `.env` values and run the cells you need.

See [`README.md`](README.md) in this folder for the full per-file walkthrough these snippets were extracted from.


## 1. Shared setup utilities

Used at the top of almost every lab file.

**Clear the console cross-platform** — appears in nearly every lab (Labs 3, 6, 12–16).

In [1]:
import os

os.system('cls' if os.name == 'nt' else 'clear')


0

**Load configuration from `.env`** — the pattern behind every `os.getenv(...)` call in this folder (variable names differ per lab — see `README.md`'s env-var table).

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")   # substitute whichever variable name the lab you're working from uses


## 2. Authentication patterns

Every lab authenticates via your `az login` session (Entra ID) — never a hardcoded API key.

**Bearer token provider for an OpenAI-compatible client** (sync) — Lab 3 (all 4 variants), Lab 14 (both files).

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), "https://ai.azure.com/.default"
)


**Bearer token provider, async version** (`azure.identity.aio`) — Lab 3 `chat-async.py`. The credential must be explicitly closed when you're done with it.

In [ ]:
from azure.identity.aio import DefaultAzureCredential, get_bearer_token_provider

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential, "https://ai.azure.com/.default"
)

# ... use the credential ...

await credential.close()


**Plain `DefaultAzureCredential` for a Foundry project client** (no token provider needed — the SDK client accepts the credential object directly) — Labs 6, 7, 8, 9, 12, 13, 16.

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()


**`DefaultAzureCredential` with specific auth methods excluded** — Lab 9. Skips environment-variable and managed-identity auth, forcing your `az login` (Azure CLI) session to be used.

In [ ]:
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential(
    exclude_environment_credential=True,
    exclude_managed_identity_credential=True
)


**`AzureCliCredential`** — an alternative to `DefaultAzureCredential` that goes straight to your Azure CLI session, used by the Microsoft Agent Framework labs (10, 11) and the async VoiceLive lab (17).

In [ ]:
from azure.identity import AzureCliCredential

credential = AzureCliCredential()


## 3. Generative AI — Azure OpenAI clients & chat calls (Lab 3–4)

**`OpenAI` client pointed at an Azure endpoint** (sync) — the OpenAI-compatible way to reach Azure OpenAI through Foundry. Lab 3 (`chat-app-chatcompletion.py`, `chat-app-responseapi.py`, `chat-app-responseapi-stream.py`).

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url=azure_openai_endpoint,
    api_key=token_provider
)


**`AsyncOpenAI` client** (async equivalent of the above) — Lab 3 `chat-async.py`.

In [ ]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    base_url=azure_openai_endpoint,
    api_key=token_provider
)


**`AzureOpenAI` client for audio models** (TTS / transcription) — Lab 14 (both files). Needs an explicit `api_version`, unlike the plain `OpenAI` client above.

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2025-03-01-preview"
)


**Chat Completions API call** (the classic/older shape — you resend the whole `messages` list every turn) — Lab 3 `chat-app-chatcompletion.py`.

In [ ]:
completion = openai_client.chat.completions.create(
    model=model_deployment,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant that answers questions and provides information."},
        {"role": "user", "content": input_text}
    ]
)
print(completion.choices[0].message.content)


**Responses API call with conversation memory** (the newer shape — no message list, memory tracked server-side via `previous_response_id`) — Lab 3 `chat-app-responseapi.py`.

In [ ]:
response = openai_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id,
)
print(response.output_text)
last_response_id = response.id


**Responses API call, streamed token-by-token** — Lab 3 `chat-app-responseapi-stream.py`.

In [ ]:
stream = openai_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id,
    stream=True
)
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="")
    elif event.type == "response.completed":
        last_response_id = event.response.id


**Responses API call, async/await** — Lab 3 `chat-async.py`.

In [ ]:
response = await async_client.responses.create(
    model=model_deployment,
    instructions="You are a helpful AI assistant that answers questions and provides information.",
    input=input_text,
    previous_response_id=last_response_id
)
assistant_text = response.output_text
last_response_id = response.id


## 4. Foundry Agent Service — connecting & conversations (Lab 6–9, 12–13, 16)

**Connect to a Foundry project (plain-variable style)** — Labs 6, 9, 12, 13, 16.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


**Connect to a Foundry project (context-manager style)** — closes the credential/clients automatically. Labs 7, 8.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as openai_client,
):
    ...  # use project_client / openai_client inside this block


**Look up an agent already created in the Foundry portal** (by name — this code never creates or edits the agent) — Labs 6, 9, 13, 16.

In [ ]:
agent = project_client.agents.get(agent_name=agent_name)


**Create a new agent version in code** (`PromptAgentDefinition`) — Labs 7, 8, 10 (Agent Framework equivalent shown separately below).

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name="astronomy-agent",
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions="You are an astronomy observations assistant...",
        tools=[event_tool, cost_tool, report_tool],   # omit `tools=` entirely for a plain, tool-less agent
    ),
)


**Delete an agent version** (cleanup, for agents your own script created) — Labs 7, 8.

In [ ]:
project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)


**Create a conversation** (server-side message-history container) — Labs 6–9, 12–13, 16.

In [ ]:
conversation = openai_client.conversations.create()
# or, equivalently: openai_client.conversations.create(items=[])


**Add a user message to a conversation**

In [ ]:
openai_client.conversations.items.create(
    conversation_id=conversation.id,
    items=[{"type": "message", "role": "user", "content": user_input}],
)


**Get an agent's reply, continuing an existing conversation** (`input=""` — the message was already added to the conversation above) — Labs 6, 9.

In [ ]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    input=""
)
print(response.output_text)


**Get an agent's reply, one-shot with no conversation object** (no memory across calls) — Labs 13, 16.

In [ ]:
response = openai_client.responses.create(
    input=[{"role": "user", "content": prompt}],
    extra_body={"agent_reference": {"name": agent_name, "type": "agent_reference"}},
)
print(f"{agent_name}: {response.output_text}")


## 5. Function calling / tools (Lab 7–8)

**Define a `FunctionTool`** (JSON-schema description of a Python function so the model knows it exists) — Lab 7 (three of these, one per tool — shown here is the representative shape; parameters differ per tool).

In [ ]:
from azure.ai.projects.models import FunctionTool

event_tool = FunctionTool(
    name="next_visible_event",
    description="Get the next visible event in a given location.",
    parameters={
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "continent to find the next visible event in (e.g. 'north_america', 'south_america', 'australia')",
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    strict=True,
)


**Process `function_call` items and dispatch by name (hardcoded if/elif)** — Lab 7 `agent.py`.

In [ ]:
import json
from openai.types.responses.response_input_param import FunctionCallOutput

for item in response.output:
    if item.type == "function_call":
        result = None
        if item.name == "next_visible_event":
            result = next_visible_event(**json.loads(item.arguments))
        elif item.name == "calculate_observation_cost":
            result = calculate_observation_cost(**json.loads(item.arguments))
        elif item.name == "generate_observation_report":
            result = generate_observation_report(**json.loads(item.arguments))

        input_list.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=result,
            )
        )


**Process `function_call` items and dispatch via a lookup dict** (generalized — works for any number of dynamically-discovered tools) — Lab 8 `client.py`.

In [ ]:
import json

for item in response.output:
    if item.type == "function_call":
        kwargs = json.loads(item.arguments)
        required_function = functions_dict.get(item.name)
        output = await required_function(**kwargs)

        input_list.append(
            FunctionCallOutput(
                type="function_call_output",
                call_id=item.call_id,
                output=output.content[0].text,
            )
        )


**Send function call outputs back to the model to get its final answer**

In [ ]:
if input_list:
    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
print(f"AGENT: {response.output_text}")


## 6. MCP — Model Context Protocol (Lab 8)

**Remote/hosted MCP tool with mandatory human approval** — Lab 8 `agent.py`.

In [ ]:
from azure.ai.projects.models import MCPTool

mcp_tool = MCPTool(
    server_label="api-specs",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="always",
)


**Approve pending MCP tool calls in a loop until none remain**

In [ ]:
from openai.types.responses.response_input_param import McpApprovalResponse

while True:
    input_list = []
    for item in response.output:
        if item.type == "mcp_approval_request":
            input_list.append(
                McpApprovalResponse(
                    type="mcp_approval_response",
                    approve=True,
                    approval_request_id=item.id,
                )
            )

    if not input_list:
        break

    response = openai_client.responses.create(
        input=input_list,
        previous_response_id=response.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )


**Build a local MCP server with `fastmcp`** — Lab 8 `server.py`.

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(name="Inventory")

@mcp.tool()
def get_inventory_levels() -> dict:
    """Returns current inventory for all products."""
    return {"Moisturizer": 6, "Shampoo": 8}

mcp.run(show_banner=False)


**Connect to a local MCP server as a subprocess (stdio transport)** — Lab 8 `client.py`.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="python", args=["server.py"], env=None)

stdio_transport = await exit_stack.enter_async_context(stdio_client(server_params))
stdio, write = stdio_transport

session = await exit_stack.enter_async_context(ClientSession(stdio, write))
await session.initialize()

response = await session.list_tools()
tools = response.tools


**Wrap MCP tools as Foundry `FunctionTool`s dynamically** (bridges any MCP server's tools into an agent) — Lab 8 `client.py`.

In [ ]:
from azure.ai.projects.models import FunctionTool

def make_tool_func(tool_name):
    async def tool_func(**kwargs):
        return await session.call_tool(tool_name, kwargs)
    tool_func.__name__ = tool_name
    return tool_func

functions_dict = {tool.name: make_tool_func(tool.name) for tool in tools}

mcp_function_tools = [
    FunctionTool(
        name=tool.name,
        description=tool.description,
        parameters={"type": "object", "properties": {}, "additionalProperties": False},
        strict=True,
    )
    for tool in tools
]


## 7. Microsoft Agent Framework (Lab 10–11)

**Create a Foundry-backed chat client**

In [ ]:
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

client = FoundryChatClient(
    project_endpoint=os.getenv("PROJECT_ENDPOINT"),
    model=os.getenv("MODEL_DEPLOYMENT_NAME"),
    credential=AzureCliCredential()
)


**Define an agent tool with the `@tool` decorator** (schema is derived automatically from type hints — no hand-written JSON schema) — Lab 10.

In [ ]:
from agent_framework import tool
from pydantic import Field
from typing import Annotated

@tool(approval_mode="never_require")
def submit_claim(
    to: Annotated[str, Field(description="Who to send the email to")],
    subject: Annotated[str, Field(description="The subject of the email.")],
    body: Annotated[str, Field(description="The text body of the email.")]):
        print("\nTo:", to)
        print("Subject:", subject)
        print(body, "\n")


**Create and run an agent as an async context manager** — Lab 10.

In [ ]:
from agent_framework import Agent

async with (
    Agent(
        client=client,
        name="ExpenseClaimAgent",
        instructions="You are an AI assistant for expense claim submission...",
        tools=[submit_claim],
    ) as agent,
):
    response = await agent.run(["some prompt"])
    print(response)


**Create an agent with the `as_agent()` shorthand** (no tools needed) — Lab 11.

In [ ]:
summarizer_agent = chat_client.as_agent(
    name="summarizer",
    instructions="Summarize the customer's feedback in one short sentence.",
)


**Sequential multi-agent orchestration** (output of one agent feeds the next) — Lab 11.

In [ ]:
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(
    participants=[summarizer_agent, classifier_agent, action_agent],
    output_from="all",
).build()

result = await workflow.run("Customer feedback: ...")
outputs = result.get_outputs()


## 8. Azure AI Language — Text Analytics (Lab 12)

**Create a `TextAnalyticsClient`**

In [ ]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.identity import DefaultAzureCredential

ai_client = TextAnalyticsClient(endpoint=foundry_endpoint, credential=DefaultAzureCredential())


**Detect language, extract entities, and redact PII**

In [ ]:
detected_language = ai_client.detect_language(documents=[text])[0]
print(detected_language.primary_language.name)

entities = ai_client.recognize_entities(documents=[text])[0].entities
for entity in entities:
    print(f"{entity.text} ({entity.category})")

pii_result = ai_client.recognize_pii_entities(documents=[text])[0]
for pii_entity in pii_result.entities:
    print(f"{pii_entity.text} ({pii_entity.category})")
print(pii_result.redacted_text)


## 9. Azure AI Speech & Azure OpenAI audio models (Lab 14–16)

**Text-to-speech via an Azure OpenAI audio model, streamed to a file** — Lab 14 `generate-speech.py`.

In [ ]:
with client.audio.speech.with_streaming_response.create(
    model=model_deployment,
    voice="alloy",
    input="My voice is my passport!",
    instructions="Speak in a serious tone.",
) as response:
    response.stream_to_file(speech_file_path)


**Transcription via an Azure OpenAI audio model** — Lab 14 `transcribe-speech.py`.

In [ ]:
audio_file = open(file_path, "rb")
transcription = client.audio.transcriptions.create(
    model=model_deployment,
    file=audio_file,
    response_format="text"
)
print(transcription)


**`SpeechConfig` authenticated via Entra ID** (dedicated Azure AI Speech SDK, no subscription key) — Lab 15.

In [ ]:
import azure.cognitiveservices.speech as speech_sdk
from azure.identity import DefaultAzureCredential

speech_config = speech_sdk.SpeechConfig(
    token_credential=DefaultAzureCredential(),
    endpoint=foundry_endpoint
)


**Text-to-speech to a file with the dedicated Speech SDK** — Lab 15 `record_greeting()`.

In [ ]:
output_file = "greeting.wav"
audio_config = speech_sdk.audio.AudioOutputConfig(filename=output_file)
speech_config.speech_synthesis_voice_name = "en-US-Serena:DragonHDLatestNeural"

speech_synthesizer = speech_sdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_config)
result = speech_synthesizer.speak_text_async(greeting_message).get()

if result.reason == speech_sdk.ResultReason.SynthesizingAudioCompleted:
    print(f"Greeting recorded and saved to {output_file}")


**Transcribe an audio file with the dedicated Speech SDK** — Lab 15 `transcribe_messages()`.

In [ ]:
audio_config = speech_sdk.audio.AudioConfig(filename=file_path)
speech_recognizer = speech_sdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)
result = speech_recognizer.recognize_once_async().get()

if result.reason == speech_sdk.ResultReason.RecognizedSpeech:
    print(f"Transcription: {result.text}")


## 10. Azure AI VoiceLive — real-time voice agent (Lab 17)

**Connect to VoiceLive and configure a real-time session**

In [ ]:
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import AgentConfig, RequestSession, Modality, InputAudioFormat, OutputAudioFormat, AzureSemanticVadMultilingual, AudioEchoCancellation, AudioNoiseReduction

agent_config = AgentConfig({"agent_name": agent_name, "project_name": project_name})

async with connect(
    endpoint=endpoint,
    credential=credential,
    api_version="2026-01-01-preview",
    agent_config=agent_config
) as connection:
    session_config = RequestSession(
        modalities=[Modality.TEXT, Modality.AUDIO],
        input_audio_format=InputAudioFormat.PCM16,
        output_audio_format=OutputAudioFormat.PCM16,
        turn_detection=AzureSemanticVadMultilingual(),
        input_audio_echo_cancellation=AudioEchoCancellation(),
        input_audio_noise_reduction=AudioNoiseReduction(type="azure_deep_noise_suppression"),
    )
    await connection.session.update(session=session_config)


**Process streamed server events** — the main real-time event loop.

In [ ]:
from azure.ai.voicelive.models import ServerEventType

async for event in connection:
    if event.type == ServerEventType.SESSION_UPDATED:
        start_microphone_capture()
    elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
        queue_audio_for_playback(event.delta)
    elif event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
        clear_playback_queue()   # barge-in: stop the agent talking over the user
    elif event.type == ServerEventType.ERROR:
        print(f"Error: {event.error.message}")


**Capture microphone audio and stream it to VoiceLive** (PyAudio input callback).

In [ ]:
import base64
import pyaudio

def capture_callback(in_data, frame_count, time_info, status):
    audio_base64 = base64.b64encode(in_data).decode("utf-8")
    asyncio.run_coroutine_threadsafe(
        connection.input_audio_buffer.append(audio=audio_base64),
        event_loop
    )
    return (None, pyaudio.paContinue)

input_stream = audio.open(
    format=pyaudio.paInt16, channels=1, rate=24000,
    input=True, frames_per_buffer=1200,
    stream_callback=capture_callback
)


**Play received audio through speakers, buffered via a queue** (PyAudio output callback).

In [ ]:
import queue
import pyaudio

playback_queue = queue.Queue()
remaining = bytes()

def playback_callback(in_data, frame_count, time_info, status):
    global remaining
    bytes_needed = frame_count * pyaudio.get_sample_size(pyaudio.paInt16)
    output = remaining[:bytes_needed]
    remaining = remaining[bytes_needed:]

    while len(output) < bytes_needed:
        try:
            audio_data = playback_queue.get_nowait()
            if audio_data is None:
                break
            output += audio_data
        except queue.Empty:
            output += bytes(bytes_needed - len(output))  # pad with silence
            break

    return (output, pyaudio.paContinue)


## 11. File / citation handling helpers (Lab 6)

Helpers for saving files an agent's code-interpreter tool generates or cites, since the response only contains a *reference* to sandboxed files, not the bytes themselves.

**Save binary/base64 data to a uniquely-named local file**

In [ ]:
import base64
from pathlib import Path

OUTPUT_DIR = Path("agent_outputs")

def get_output_path(filename):
    OUTPUT_DIR.mkdir(exist_ok=True)
    file_name = Path(filename).name
    stem, suffix = Path(file_name).stem or "output", Path(file_name).suffix
    output_path = OUTPUT_DIR / file_name
    counter = 1
    while output_path.exists():
        output_path = OUTPUT_DIR / f"{stem}_{counter}{suffix}"
        counter += 1
    return output_path

def save_bytes(file_bytes, filename):
    output_path = get_output_path(filename)
    with open(output_path, "wb") as f:
        f.write(file_bytes)
    return output_path

def save_image(image_data, filename):
    return save_bytes(base64.b64decode(image_data), filename)


**Download a file cited by the agent's code-interpreter sandbox**

In [ ]:
def download_container_file(openai_client, annotation, downloaded_files):
    cache_key = (annotation.container_id, annotation.file_id)
    if cache_key in downloaded_files:
        return downloaded_files[cache_key]

    file_content = openai_client.containers.files.content.retrieve(
        file_id=annotation.file_id,
        container_id=annotation.container_id,
    )
    output_path = save_bytes(file_content.read(), annotation.filename or f"{annotation.file_id}.bin")
    downloaded_files[cache_key] = output_path
    return output_path


## 12. Prompt engineering & LLM-as-judge evaluation (Lab 5)

New in the "fill all missing" pass — Labs 18–28 close AI-103 gaps that had no hands-on lab coverage before. See `README.md`'s "AI-103 exam coverage" table for which of these are proven vs. best-effort/preview.

**Few-shot prompt** — worked examples baked into the prompt for a consistent output format, instead of a bare zero-shot question.

In [ ]:
few_shot_prompt = """Classify the sentiment of a product review as exactly one word: Positive, Negative, or Mixed.

Example 1:
Review: "This blender is amazing, it crushes ice in seconds!"
Sentiment: Positive

Example 2:
Review: "The battery died after two days and support never replied."
Sentiment: Negative

Now classify this review:
Review: "{review_text}"
Sentiment:"""


**LLM-as-judge: draft → critique → conditional regenerate** — the real pattern this repo uses for "evaluation" (no `azure-ai-evaluation` SDK anywhere in this repo).

In [ ]:
# Pass 1: draft answer
draft_response = openai_client.responses.create(model=model_deployment, input=question)
answer = draft_response.output_text

# Pass 2: judge call - same client, a grading prompt, a strict output convention
critique_response = openai_client.responses.create(
    model=model_deployment,
    input=(
        f"You are reviewing an AI assistant's answer for completeness.\n\n"
        f"Question: {question}\n\nAnswer: {answer}\n\n"
        "Reply with only COMPLETE or MISSING."
    ),
)
critique = critique_response.output_text

# Pass 3: only regenerate if the judge actually flagged a problem
if "MISSING" in critique.upper():
    improved = openai_client.responses.create(
        model=model_deployment,
        input=f"Answer this question completely, addressing every part: {question}",
    )
    answer = improved.output_text


## 13. Multi-agent delegation — `ConnectedAgentTool` (Lab 18)

Different from section 7's `SequentialBuilder` (pipes agents in sequence) — here ONE orchestrator agent calls other already-created agents AS TOOLS, on demand, in whatever order the conversation needs.

**Create specialist agents, then wrap each as a `ConnectedAgentTool`** — note `AgentsClient` (not `AIProjectClient`) and the older thread/run model.

In [ ]:
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import ConnectedAgentTool

agents_client = AgentsClient(endpoint=project_endpoint, credential=DefaultAzureCredential())

specialist_agent = agents_client.create_agent(
    model=model_deployment, name="availability-checker", instructions="...",
)

specialist_tool = ConnectedAgentTool(
    id=specialist_agent.id,
    name="check_availability",
    description="Check whether a room type is available for given dates.",
)

# .definitions (plural, a LIST) - not the tool object itself - goes into tools=
orchestrator = agents_client.create_agent(
    model=model_deployment,
    name="concierge",
    instructions="Use your connected agents to answer the guest, combining their replies.",
    tools=specialist_tool.definitions,
)


**Run a thread against the orchestrator** — delegation to connected agents happens automatically inside `create_and_process()`, no manual function-call loop needed.

In [ ]:
thread = agents_client.threads.create()
agents_client.messages.create(thread_id=thread.id, role="user", content=user_question)
agents_client.runs.create_and_process(thread_id=thread.id, agent_id=orchestrator.id)

reply = agents_client.messages.get_last_message_text_by_role(thread_id=thread.id, role="assistant")
print(reply.text.value)

agents_client.delete_agent(orchestrator.id)
agents_client.delete_agent(specialist_agent.id)


## 14. Preview / best-effort patterns (Lab 19–21, 27)

⚠️ None of the class/method names below are confirmed against a real, current SDK — each script wraps its guessed import/call so it explains the concept instead of crashing uninformatively if the surface doesn't match. See each lab's file header for full caveats.

**Foundry IQ (shared knowledge platform)** — a knowledge source registered once, referenced by many agents (Lab 19). `KnowledgeTool` is a guessed class name, modeled on the real `MCPTool`/`FunctionTool` shape.

In [ ]:
try:
    from azure.ai.projects.models import KnowledgeTool  # ⚠️ guessed — not confirmed to exist
    knowledge_tool = KnowledgeTool(knowledge_source_name=knowledge_source_name)
    # Any number of agents can attach this SAME tool and all ground/cite consistently
except ImportError:
    print("KnowledgeTool not available on this SDK version - check current Foundry IQ docs.")


**A2A Agent Card** — the real, public discovery-document shape A2A agents publish (unlike the SDK class names elsewhere in this section, this JSON shape IS documented by the open A2A spec) — Lab 21.

In [ ]:
agent_card = {
    "name": "astronomy-agent",
    "description": "Looks up astronomical events and calculates telescope rental costs.",
    "url": "https://example.com/agents/astronomy",
    "skills": [
        {"id": "next_visible_event", "name": "Next visible event", "description": "..."},
    ],
}


**Sora 2 video generation — async submit-then-poll job** (Lab 27). Modeled on OpenAI's public video-generation API pattern; not confirmed on Azure OpenAI.

In [ ]:
video_job = client.videos.create(model=video_deployment, prompt="...", size="1280x720", seconds="4")

while video_job.status in ("queued", "processing", "in_progress"):
    time.sleep(5)
    video_job = client.videos.retrieve(video_job.id)

content = client.videos.download_content(video_job.id, variant="video")
content.write_to_file("generated_video.mp4")


## 15. Image generation & editing (Lab 22)

**Generate an image from a text prompt**

In [ ]:
response = client.images.generate(
    model=image_deployment,
    prompt="A professional product photo of a blue ceramic coffee mug on a wooden table.",
    n=1,
    size="1024x1024",
    quality="medium",
    output_format="png",
)
image_bytes = base64.b64decode(response.data[0].b64_json)
Path("generated_image.png").write_bytes(image_bytes)


**Edit an existing image with a text instruction** (whole image may change)

In [ ]:
with open("input_image.png", "rb") as image_file:
    response = client.images.edit(
        model=image_deployment,
        image=image_file,
        prompt="Make the lighting warmer and add a soft bokeh background.",
        size="1024x1024",
        n=1,
    )


**Masked edit / inpainting** (only the masked region changes)

In [ ]:
with open("input_image.png", "rb") as image_file, open("mask.png", "rb") as mask_file:
    response = client.images.edit(
        model=image_deployment,
        image=image_file,
        mask=mask_file,   # transparent pixels mark the region to replace
        prompt="Replace the background with a bright blue sky.",
        size="1024x1024",
        n=1,
    )


## 16. Content Safety, Content Understanding & Document Intelligence (Lab 23–25)

**`ContentSafetyClient` — moderate text**, one severity score (0–7) per harm category, not a single pass/fail flag.

In [ ]:
from azure.ai.contentsafety import ContentSafetyClient
from azure.ai.contentsafety.models import AnalyzeTextOptions
from azure.core.credentials import AzureKeyCredential

client = ContentSafetyClient(endpoint=endpoint, credential=AzureKeyCredential(key))

result = client.analyze_text(AnalyzeTextOptions(text=text))
for category_result in result.categories_analysis:
    print(category_result.category, category_result.severity)


**`ContentSafetyClient` — moderate an image** (same client, `AnalyzeImageOptions` + raw bytes).

In [ ]:
from azure.ai.contentsafety.models import AnalyzeImageOptions, ImageData

result = client.analyze_image(AnalyzeImageOptions(image=ImageData(content=image_bytes)))


**`ContentUnderstandingClient`** ⚠️ (unverified/preview — package not installed in this repo; two Section Code files call this with different method names for the same operation) — Lab 24.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput

client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key))
poller = client.begin_analyze(analyzer_id="prebuilt-invoice", inputs=[AnalysisInput(url=document_url)])
result = poller.result()
print(result.as_dict())


**`DocumentIntelligenceClient`** (proven, stable SDK) — extracts text/tables from a document as a long-running operation — Lab 25.

In [ ]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key), api_version="2024-11-30")

poller = client.begin_analyze_document(
    model_id="prebuilt-layout",
    body=AnalyzeDocumentRequest(url_source=document_url),
    output_content_format="markdown",
)
result = poller.result()
print(result.content)   # extracted text as markdown
for table in result.tables:
    print(table.row_count, table.column_count)


## 17. Azure AI Search — bring-your-own-retrieval RAG (Lab 26)

**Hybrid keyword + vector search**, merged server-side via Reciprocal Rank Fusion (RRF).

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery

search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=DefaultAzureCredential())

results = search_client.search(
    search_text=question,                                                  # keyword side
    vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=3, fields="text_vector")],  # vector side
    select=["chunk", "title"],
    top=3,
)
sources = "\n\n".join(f"Source ({r['title']}):\n{r['chunk']}" for r in results)


**Manually ground a prompt in the retrieved chunks** ("bring your own retrieval" — no Foundry-native search tool involved).

In [ ]:
prompt = f"""Answer using ONLY the sources below. If they don't contain the answer, say you don't know.

{sources}

Question: {question}
Answer:"""

response = openai_client.responses.create(model=model_deployment, input=prompt)


## 18. Translation — LLM-prompted vs. dedicated service (Lab 28)

**LLM-prompted translation** — flexible instructions, one call per target language.

In [ ]:
response = openai_client.responses.create(
    model=model_deployment,
    input=[
        {"type": "message", "role": "system", "content": f"You are a professional translator. Translate into {target_language}. Reply with ONLY the translated text."},
        {"type": "message", "role": "user", "content": text},
    ],
)
translated = response.output_text


**Azure AI Translator — dedicated REST service**, one call can target MANY languages at once (no Python SDK package in common use — plain `requests`).

In [ ]:
import requests

headers = {"Ocp-Apim-Subscription-Key": key, "Content-Type": "application/json"}
url = f"{endpoint}translator/text/translate?api-version=2025-10-01-preview"
body = {"inputs": [{"Text": text, "language": "en", "targets": [{"language": "fr"}, {"language": "ja"}]}]}

response = requests.post(url, headers=headers, json=body)
response.raise_for_status()
for t in response.json()["value"][0]["translations"]:
    print(t["language"], t["text"])
